# GPU Ensemble Energy Forecasting

Kaggle-ready notebook for GPU training. It runs everything one-by-one:

1. Load the prepared CSV
2. Split chronologically
3. Tune XGBoost on GPU
4. Tune LightGBM on GPU
5. Tune CatBoost on GPU
6. Build a weighted ensemble
7. Report MSE, MAE, R2, and MASE

In Kaggle, set Accelerator to GPU before running.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import ParameterSampler, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print('XGBoost unavailable:', exc)

try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except Exception as exc:
    HAS_LIGHTGBM = False
    print('LightGBM unavailable:', exc)

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception as exc:
    HAS_CATBOOST = False
    print('CatBoost unavailable:', exc)

RANDOM_STATE = 42
TARGET = 'consumption'
TIME_COL = 'start_time'
SEASONAL_PERIOD = 24
CV_SPLITS = 3
N_ITER_PER_MODEL = 10  # increase to 25-50 in Kaggle if you have time

np.random.seed(RANDOM_STATE)

## 1. Load CSV

In [ ]:
# Upload cleaned_energy_data_model_minus_first_week_with_features.csv as a Kaggle dataset.
# If auto-detection fails, paste the exact Kaggle path into INPUT_PATH.
INPUT_PATH = None

if INPUT_PATH is None:
    matches = list(Path('/kaggle/input').rglob('cleaned_energy_data_model_minus_first_week_with_features.csv'))
    INPUT_PATH = matches[0] if matches else Path('cleaned_energy_data_model_minus_first_week_with_features.csv')

print('Using input file:', INPUT_PATH)
df = pd.read_csv(INPUT_PATH)
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce')
df = df.dropna(subset=[TIME_COL, TARGET]).sort_values(TIME_COL).reset_index(drop=True)

print('Rows, columns:', df.shape)
display(df.head())

## 2. Chronological Split

In [ ]:
drop_cols = {TARGET, TIME_COL, 'end_time_utc'}
feature_cols = [
    c for c in df.columns
    if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])
]

model_df = df[[TIME_COL, TARGET] + feature_cols].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

test_idx = int(len(model_df) * 0.80)
train_val_df = model_df.iloc[:test_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx:].reset_index(drop=True)

val_idx = int(len(train_val_df) * 0.85)
tune_train_df = train_val_df.iloc[:val_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx:].reset_index(drop=True)

X_tune = tune_train_df[feature_cols]
y_tune = tune_train_df[TARGET]
X_val = val_df[feature_cols]
y_val = val_df[TARGET]
X_train_val = train_val_df[feature_cols]
y_train_val = train_val_df[TARGET]
X_test = test_df[feature_cols]
y_test = test_df[TARGET]

print('Feature count:', len(feature_cols))
print('Tune train:', tune_train_df[TIME_COL].min(), 'to', tune_train_df[TIME_COL].max(), len(tune_train_df))
print('Validation:', val_df[TIME_COL].min(), 'to', val_df[TIME_COL].max(), len(val_df))
print('Test:', test_df[TIME_COL].min(), 'to', test_df[TIME_COL].max(), len(test_df))

## 3. Metrics

In [ ]:
def mase(y_true, y_pred, y_train, seasonal_period=24):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    naive_errors = np.abs(y_train[seasonal_period:] - y_train[:-seasonal_period])
    scale = naive_errors.mean()
    if scale == 0 or np.isnan(scale):
        return np.nan
    return np.abs(y_true - y_pred).mean() / scale

def regression_metrics(y_true, y_pred, y_train_for_mase):
    return {
        'MSE': float(mean_squared_error(y_true, y_pred)),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred)),
        'MASE': float(mase(y_true, y_pred, y_train_for_mase, seasonal_period=SEASONAL_PERIOD)),
    }

def print_metrics(name, metrics):
    print(f'\n{name}')
    for key, value in metrics.items():
        print(f'{key}: {value:.6f}')

## 4. One-By-One GPU Tuning Helper

In [ ]:
def tune_model_one_by_one(model_name, build_model, param_grid, X, y, n_iter=N_ITER_PER_MODEL):
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    sampled_params = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=RANDOM_STATE))
    rows = []
    best_params = None
    best_mse = np.inf

    print(f'\n===== Tuning {model_name} on GPU, one candidate at a time =====')
    for candidate_idx, params in enumerate(sampled_params, start=1):
        fold_mses = []
        print(f'\n{model_name} candidate {candidate_idx}/{len(sampled_params)}')
        print(params)

        for fold_idx, (train_idx, valid_idx) in enumerate(tscv.split(X), start=1):
            X_train_fold = X.iloc[train_idx]
            y_train_fold = y.iloc[train_idx]
            X_valid_fold = X.iloc[valid_idx]
            y_valid_fold = y.iloc[valid_idx]

            model = build_model(params)
            model.fit(X_train_fold, y_train_fold)
            pred = model.predict(X_valid_fold)
            fold_mse = mean_squared_error(y_valid_fold, pred)
            fold_mses.append(fold_mse)
            print(f'  fold {fold_idx} MSE: {fold_mse:.4f}')

            del model
            gc.collect()

        mean_mse = float(np.mean(fold_mses))
        row = {'model': model_name, 'candidate': candidate_idx, 'cv_mse': mean_mse, 'params': params}
        rows.append(row)
        print(f'  mean CV MSE: {mean_mse:.4f}')

        if mean_mse < best_mse:
            best_mse = mean_mse
            best_params = params
            print('  new best')

    return best_params, pd.DataFrame(rows).sort_values('cv_mse').reset_index(drop=True)

## 5. Tune XGBoost GPU

In [ ]:
best_models = {}
tuning_tables = []

def build_xgb(params):
    return XGBRegressor(
        objective='reg:squarederror',
        tree_method='hist',
        device='cuda',
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbosity=1,
        **params,
    )

xgb_grid = {
    'n_estimators': [700, 1000, 1300, 1600],
    'learning_rate': [0.02, 0.03, 0.04, 0.05],
    'max_depth': [3, 4, 5, 6],
    'min_child_weight': [1, 2, 3, 5],
    'subsample': [0.8, 0.85, 0.9, 0.95],
    'colsample_bytree': [0.8, 0.85, 0.9, 0.95],
    'reg_alpha': [0.0, 0.01, 0.02, 0.05],
    'reg_lambda': [1.0, 1.5, 2.0, 3.0],
}

if HAS_XGBOOST:
    xgb_best_params, xgb_tuning = tune_model_one_by_one('xgboost_gpu', build_xgb, xgb_grid, X_tune, y_tune)
    tuning_tables.append(xgb_tuning)
    best_models['xgboost_gpu'] = build_xgb(xgb_best_params)
    print('Best XGBoost GPU params:', xgb_best_params)
else:
    print('Skipping XGBoost because package is unavailable.')

## 6. Tune LightGBM GPU

In [ ]:
def build_lgbm(params):
    return LGBMRegressor(
        objective='regression',
        device='gpu',
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=-1,
        **params,
    )

lgbm_grid = {
    'n_estimators': [700, 1000, 1300, 1600],
    'learning_rate': [0.02, 0.03, 0.04, 0.05],
    'num_leaves': [31, 63, 127, 255],
    'max_depth': [-1, 5, 7, 9],
    'min_child_samples': [10, 20, 40, 80],
    'subsample': [0.8, 0.85, 0.9, 0.95],
    'colsample_bytree': [0.8, 0.85, 0.9, 0.95],
    'reg_alpha': [0.0, 0.01, 0.05, 0.1],
    'reg_lambda': [0.0, 0.5, 1.0, 2.0],
}

if HAS_LIGHTGBM:
    lgbm_best_params, lgbm_tuning = tune_model_one_by_one('lightgbm_gpu', build_lgbm, lgbm_grid, X_tune, y_tune)
    tuning_tables.append(lgbm_tuning)
    best_models['lightgbm_gpu'] = build_lgbm(lgbm_best_params)
    print('Best LightGBM GPU params:', lgbm_best_params)
else:
    print('Skipping LightGBM because package is unavailable.')

## 7. Tune CatBoost GPU

In [ ]:
def build_catboost(params):
    return CatBoostRegressor(
        loss_function='RMSE',
        task_type='GPU',
        devices='0',
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        **params,
    )

cat_grid = {
    'iterations': [700, 1000, 1300, 1600],
    'learning_rate': [0.02, 0.03, 0.04, 0.05],
    'depth': [4, 5, 6, 7, 8],
    'l2_leaf_reg': [1, 3, 5, 7, 9],
    'bagging_temperature': [0.0, 0.25, 0.5, 1.0],
    'random_strength': [0.0, 0.25, 0.5, 1.0],
}

if HAS_CATBOOST:
    cat_best_params, cat_tuning = tune_model_one_by_one('catboost_gpu', build_catboost, cat_grid, X_tune, y_tune)
    tuning_tables.append(cat_tuning)
    best_models['catboost_gpu'] = build_catboost(cat_best_params)
    print('Best CatBoost GPU params:', cat_best_params)
else:
    print('Skipping CatBoost because package is unavailable.')

## 8. Validation Predictions and Ensemble Weights

In [ ]:
if not best_models:
    raise RuntimeError('No GPU-capable models were available. Check Kaggle packages and GPU accelerator.')

val_preds = {}
val_rows = []

for name, model in best_models.items():
    print(f'\nFitting {name} on tune_train and predicting validation...')
    model.fit(X_tune, y_tune)
    pred = model.predict(X_val)
    val_preds[name] = pred
    metrics = regression_metrics(y_val, pred, y_tune)
    val_rows.append({'model': name, **metrics})
    print_metrics(name + ' validation', metrics)
    gc.collect()

val_metrics_df = pd.DataFrame(val_rows).sort_values('MSE')
display(val_metrics_df)

model_names = list(val_preds.keys())
pred_matrix = np.column_stack([val_preds[name] for name in model_names])

rng = np.random.default_rng(RANDOM_STATE)
best_weights = np.ones(len(model_names)) / len(model_names)
best_mse = mean_squared_error(y_val, pred_matrix @ best_weights)

for _ in range(5000):
    weights = rng.dirichlet(np.ones(len(model_names)))
    mse = mean_squared_error(y_val, pred_matrix @ weights)
    if mse < best_mse:
        best_mse = mse
        best_weights = weights

ensemble_weights = dict(zip(model_names, best_weights))
print('Best validation ensemble MSE:', best_mse)
print('Ensemble weights:', ensemble_weights)

## 9. Final Test Evaluation

In [ ]:
test_preds = {}
final_rows = []

for name, model in best_models.items():
    print(f'\nFinal fit for {name} on train+validation, then test prediction...')
    model.fit(X_train_val, y_train_val)
    pred = model.predict(X_test)
    test_preds[name] = pred
    metrics = regression_metrics(y_test, pred, y_train_val)
    final_rows.append({'model': name, **metrics})
    print_metrics(name + ' test', metrics)
    gc.collect()

test_matrix = np.column_stack([test_preds[name] for name in model_names])
ensemble_pred = test_matrix @ best_weights
ensemble_metrics = regression_metrics(y_test, ensemble_pred, y_train_val)
final_rows.append({'model': 'gpu_weighted_ensemble', **ensemble_metrics})
print_metrics('gpu_weighted_ensemble test', ensemble_metrics)

final_results = pd.DataFrame(final_rows).sort_values('MSE')
display(final_results)

## 10. Save Outputs

In [ ]:
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

predictions = pd.DataFrame({
    TIME_COL: test_df[TIME_COL],
    'actual_consumption': y_test,
    'ensemble_prediction': ensemble_pred,
    'ensemble_error': y_test.to_numpy() - ensemble_pred,
    'ensemble_absolute_error': np.abs(y_test.to_numpy() - ensemble_pred),
})
for name, pred in test_preds.items():
    predictions[f'{name}_prediction'] = pred

tuning_results = pd.concat(tuning_tables, ignore_index=True) if tuning_tables else pd.DataFrame()
weights_df = pd.DataFrame({'model': list(ensemble_weights.keys()), 'weight': list(ensemble_weights.values())})

predictions.to_csv(output_dir / 'gpu_ensemble_test_predictions.csv', index=False)
final_results.to_csv(output_dir / 'gpu_ensemble_model_metrics.csv', index=False)
tuning_results.to_csv(output_dir / 'gpu_ensemble_tuning_results.csv', index=False)
weights_df.to_csv(output_dir / 'gpu_ensemble_weights.csv', index=False)

summary = {
    'feature_count': len(feature_cols),
    'models_used': model_names,
    'ensemble_weights': ensemble_weights,
    'ensemble_metrics': ensemble_metrics,
}
with open(output_dir / 'gpu_ensemble_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved files to:', output_dir)
print('- gpu_ensemble_test_predictions.csv')
print('- gpu_ensemble_model_metrics.csv')
print('- gpu_ensemble_tuning_results.csv')
print('- gpu_ensemble_weights.csv')
print('- gpu_ensemble_summary.json')